# 练习实验：决策树

在本练习中，你将从头实现一棵决策树，并将其应用于判断蘑菇可食用还是有毒的分类任务。

# 大纲
- [ 1 - 软件包 ](#1)
- [ 2 - 问题陈述](#2)
- [ 3 - 数据集](#3)
  - [ 3.1 One-hot 编码的数据集](#3.1)
- [ 4 - 决策树复习](#4)
  - [ 4.1 计算熵](#4.1)
    - [ 练习 1](#ex01)
  - [ 4.2 划分数据集](#4.2)
    - [ 练习 2](#ex02)
  - [ 4.3 计算信息增益](#4.3)
    - [ 练习 3](#ex03)
  - [ 4.4 获取最佳划分](#4.4)
    - [ 练习 4](#ex04)
- [ 5 - 构建决策树](#5)

<a name="1"></a>
## 1 - 软件包

首先，运行下面的单元格，导入完成本作业所需的全部软件包。
- [NumPy](www.numpy.org) 是在 Python 中处理矩阵的基础软件包。
- [matplotlib](http://matplotlib.org) 是 Python 中著名的绘图库。
- ``utils.py`` 包含本作业的辅助函数。你无需修改该文件中的代码。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from public_tests import *

%matplotlib inline

<a name="2"></a>
## 2 - 问题陈述

假设你正在创办一家种植并销售野生蘑菇的公司。
- 由于并非所有蘑菇都可食用，你希望能够根据给定蘑菇的物理属性，判断它是可食用还是有毒
- 你有一些可用于此任务的现有数据。

你能否利用这些数据，帮助确定哪些蘑菇可以安全销售？

注意：所使用的数据集仅用于说明目的，不能作为识别可食用蘑菇的指南。



<a name="3"></a>
## 3 - 数据集

首先，你将加载此任务的数据集。收集到的数据集如下：

| 菌盖颜色 | 菌柄形状 | 独生 | 可食用 |
|:---------:|:-----------:|:--------:|:------:|
|   棕色   |   渐细  |    是   |    1   |
|   棕色   |  膨大  |    是   |    1   |
|   棕色   |  膨大  |    否    |    0   |
|   棕色   |  膨大  |    否    |    0   |
|   棕色   |   渐细  |    是   |    1   |
|    红色    |   渐细  |    是   |    0   |
|    红色    |  膨大  |    否    |    0   |
|   棕色   |  膨大  |    是   |    1   |
|    红色    |   渐细  |    否    |    1   |
|   棕色   |  膨大  |    否    |    0   |


-  你有 10 个蘑菇样本。对于每个样本，都有：
    - 三个特征
        - 菌盖颜色（`Brown` 或 `Red`），
        - 菌柄形状（`Tapering` 或 `Enlarging`），以及
        - 是否独生（`Yes` 或 `No`）
    - 标签
        - 可食用（`1` 表示可食用，`0` 表示有毒）

<a name="3.1"></a>
### 3.1 独热编码数据集
为便于实现，我们对特征进行了独热编码（将其转换为取值为 0 或 1 的特征）

| 棕色菌盖 | 渐细的菌柄形状 | 独生 | 可食用 |
|:---------:|:--------------------:|:--------:|:------:|
|     1     |           1          |     1    |    1   |
|     1     |           0          |     1    |    1   |
|     1     |           0          |     0    |    0   |
|     1     |           0          |     0    |    0   |
|     1     |           1          |     1    |    1   |
|     0     |           1          |     1    |    0   |
|     0     |           0          |     0    |    0   |
|     1     |           0          |     1    |    1   |
|     0     |           1          |     0    |    1   |
|     1     |           0          |     0    |    0   |

因此：
- `X_train` 包含每个样本的三个特征
    - 棕色（值 `1` 表示“棕色”菌盖，`0` 表示“红色”菌盖）
    - 渐细形状（值 `1` 表示“渐细的菌柄形状”，`0` 表示“膨大”的菌柄形状）
    - 独生（值 `1` 表示“是”，`0` 表示“否”）

- `y_train` 表示蘑菇是否可食用
    - `y = 1` 表示可食用
    - `y = 0` 表示有毒

In [ ]:
X_train = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_train = np.array([1,1,0,0,1,0,0,1,1,0])

#### 查看变量
让我们进一步熟悉您的数据集。  
- 一个很好的起点是直接打印每个变量，看看其中包含什么。

下面的代码会打印 `X_train` 的前几个元素以及该变量的类型。

In [ ]:
print("First few elements of X_train:\n", X_train[:5])
print("Type of X_train:",type(X_train))

现在，让我们对 `y_train` 进行同样的操作

In [ ]:
print("First few elements of y_train:", y_train[:5])
print("Type of y_train:",type(y_train))

#### 检查变量的维度

查看数据维度是熟悉数据的另一种实用方法。

请打印 `X_train` 和 `y_train` 的形状，看看数据集中有多少个训练样本。

In [ ]:
print ('The shape of X_train is:', X_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(X_train))

<a name="4"></a>
## 4 - 决策树复习

在本练习实验中，你将基于所提供的数据集构建一棵决策树。

- 回顾一下，构建决策树的步骤如下：
    - 从根节点上的所有样本开始
    - 计算所有可能特征划分的信息增益，并选择信息增益最高的特征
    - 根据所选特征拆分数据集，并创建树的左分支和右分支
    - 不断重复拆分过程，直到满足停止条件
  
  
- 在本实验中，你将实现以下函数，它们可以让你使用信息增益最高的特征，将一个节点拆分为左分支和右分支
    - 计算节点处的熵
    - 根据给定特征，将节点处的数据集拆分为左分支和右分支
    - 计算使用给定特征拆分所产生的信息增益
    - 选择使信息增益最大化的特征
    
- 然后，我们将使用你实现的辅助函数，通过反复执行拆分过程来构建决策树，直到满足停止条件
    - 在本实验中，我们选择的停止条件是将最大深度设为 2

<a name="4.1"></a>
### 4.1 计算熵

首先，你将编写一个名为 `compute_entropy` 的辅助函数，用于计算节点的熵（不纯度的度量）。
- 该函数接收一个 NumPy 数组（`y`），该数组指示该节点中的样本是可食用（`1`）还是有毒（`0`）

补全下面的 `compute_entropy()` 函数以完成以下任务：
* 计算 $p_1$，即可食用样本所占的比例（也就是 `y` 中值为 `1` 的样本）
* 然后按下式计算熵

$$H(p_1) = -p_1 \text{log}_2(p_1) - (1- p_1) \text{log}_2(1- p_1)$$
* 注意
    * 对数以 $2$ 为底
    * 为便于实现，$0\text{log}_2(0) = 0$。也就是说，如果 `p_1 = 0` 或 `p_1 = 1`，则将熵设为 `0`
    * 请务必检查节点中的数据是否为空（即 `len(y) != 0`）。如果为空，则返回 `0`
    
<a name="ex01"></a>
### 练习 1

请按照前面的说明补全 `compute_entropy()` 函数。
    
如果遇到困难，可以查看下方单元格之后给出的提示，以帮助你完成实现。

In [ ]:
# UNQ_C1
# GRADED FUNCTION: compute_entropy

def compute_entropy(y):
    """
    Computes the entropy for 
    
    Args:
       y (ndarray): Numpy array indicating whether each example at a node is
           edible (`1`) or poisonous (`0`)
       
    Returns:
        entropy (float): Entropy at that node
        
    """
    # You need to return the following variables correctly
    entropy = 0.
    
    ### START CODE HERE ###
           
    ### END CODE HERE ###        
    
    return entropy

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
    
   * 计算 `p1`：
       * 您可以使用 `y[y == 1]` 获取 `y` 中值为 `1` 的样本子集。
       * 您可以使用 `len(y)` 获取 `y` 中的样本数量。
   * 计算 `entropy`：
       * <a href="https://numpy.org/doc/stable/reference/generated/numpy.log2.html">np.log2</a> 可用于计算 NumPy 数组以 2 为底的对数。
       * 如果 `p1` 的值为 0 或 1，请务必将熵设置为 `0`。
     
    <details>
          <summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>
        
    * 下面展示了该函数整体实现的组织方式。
    ```python 
    def compute_entropy(y):
        
        # You need to return the following variables correctly
        entropy = 0.

        ### START CODE HERE ###
        if len(y) != 0:
            # Your code here to calculate the fraction of edible examples (i.e with value = 1 in y)
            p1 =

            # For p1 = 0 and 1, set the entropy to 0 (to handle 0log0)
            if p1 != 0 and p1 != 1:
                # Your code here to calculate the entropy using the formula provided above
                entropy = 
            else:
                entropy = 0. 
        ### END CODE HERE ###        

        return entropy
    ```
    
    如果您仍然没有思路，可以查看下面给出的提示，了解如何计算 `p1` 和 `entropy`。
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 p1 的提示</b></font></summary>
           &emsp; &emsp; 您可以按如下方式计算 p1：<code>p1 = len(y[y == 1]) / len(y) </code>
    </details>

     <details>
          <summary><font size="2" color="darkblue"><b>计算熵的提示</b></font></summary>
          &emsp; &emsp; 您可以按如下方式计算熵：<code>entropy = -p1 * np.log2(p1) - (1 - p1) * np.log2(1 - p1)</code>
    </details>
        
    </details>

</details>

    

可以通过运行以下测试代码检查实现是否正确：

In [ ]:
# Compute entropy at the root node (i.e. with all examples)
# Since we have 5 edible and 5 non-edible mushrooms, the entropy should be 1"

print("Entropy at root node: ", compute_entropy(y_train)) 

# UNIT TESTS
compute_entropy_test(compute_entropy)

**预期输出**：
<table>
  <tr>
    <td> <b>根节点的熵：<b> 1.0 </td> 
  </tr>
</table>

<a name="4.2"></a>
### 4.2 划分数据集

接下来，你将编写一个名为 `split_dataset` 的辅助函数。该函数接收节点处的数据和用于划分的特征，并将数据划分到左、右分支。本实验稍后会实现代码来计算划分的优劣。

- 该函数接收训练数据、该节点处数据点的索引列表，以及用于划分的特征。
- 它划分数据，并返回左分支和右分支中的索引子集。
- 例如，假设从根节点开始（即 `node_indices = [0,1,2,3,4,5,6,7,8,9]`），并选择使用特征 `0` 进行划分，该特征表示样本是否具有棕色菌盖。
    - 函数的输出将是 `left_indices = [0,1,2,3,4,7,9]` 和 `right_indices = [5,6,8]`
    
| 索引 | 棕色菌盖 | 菌柄形状渐细 | 单生 | 可食用 |
|:-----:|:---------:|:--------------------:|:--------:|:------:|
|   0   |     1     |           1          |     1    |    1   |
|   1   |     1     |           0          |     1    |    1   |
|   2   |     1     |           0          |     0    |    0   |
|   3   |     1     |           0          |     0    |    0   |
|   4   |     1     |           1          |     1    |    1   |
|   5   |     0     |           1          |     1    |    0   |
|   6   |     0     |           0          |     0    |    0   |
|   7   |     1     |           0          |     1    |    1   |
|   8   |     0     |           1          |     0    |    1   |
|   9   |     1     |           0          |     0    |    0   |

<a name="ex02"></a>
### 练习 2

请补全下面所示的 `split_dataset()` 函数

- 对 `node_indices` 中的每个索引
    - 如果该特征在 `X` 的这个索引处取值为 `1`，则将该索引添加到 `left_indices`
    - 如果该特征在 `X` 的这个索引处取值为 `0`，则将该索引添加到 `right_indices`

如果遇到困难，可以查看下方单元格之后给出的提示，以帮助你完成实现。

In [ ]:
# UNQ_C2
# GRADED FUNCTION: split_dataset

def split_dataset(X, node_indices, feature):
    """
    Splits the data at the given node into
    left and right branches
    
    Args:
        X (ndarray):             Data matrix of shape(n_samples, n_features)
        node_indices (ndarray):  List containing the active indices. I.e, the samples being considered at this step.
        feature (int):           Index of feature to split on
    
    Returns:
        left_indices (ndarray): Indices with feature value == 1
        right_indices (ndarray): Indices with feature value == 0
    """
    
    # You need to return the following variables correctly
    left_indices = []
    right_indices = []
    
    ### START CODE HERE ###
           
    ### END CODE HERE ###
        
    return left_indices, right_indices

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
    
   * 下面展示了该函数整体实现的组织方式。
    ```python 
    def split_dataset(X, node_indices, feature):
    
        # You need to return the following variables correctly
        left_indices = []
        right_indices = []

        ### START CODE HERE ###
        # Go through the indices of examples at that node
        for i in node_indices:   
            if # Your code here to check if the value of X at that index for the feature is 1
                left_indices.append(i)
            else:
                right_indices.append(i)
        ### END CODE HERE ###
        
    return left_indices, right_indices
    ```
    <details>
          <summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>
        
    条件为 <code> if X[i][feature] == 1:</code>。
        
    </details>

</details>

    

现在使用下面的代码块检查你的实现。尝试在根节点处拆分数据集；根节点包含所有样本，按我们上面讨论的特征 0（棕色菌盖）进行拆分

In [ ]:
root_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# Feel free to play around with these variables
# The dataset only has three features, so this value can be 0 (Brown Cap), 1 (Tapering Stalk Shape) or 2 (Solitary)
feature = 0

left_indices, right_indices = split_dataset(X_train, root_indices, feature)

print("Left indices: ", left_indices)
print("Right indices: ", right_indices)

# UNIT TESTS    
split_dataset_test(split_dataset)

**预期输出**：
```
Left indices:  [0, 1, 2, 3, 4, 7, 9]
Right indices:  [5, 6, 8]
```

<a name="4.3"></a>
### 4.3 计算信息增益

接下来，你将编写一个名为 `information_gain` 的函数，它接收训练数据、某个节点上的索引以及用于划分的特征，并返回该划分产生的信息增益。

<a name="ex03"></a>
### 练习 3

请完成下面所示的 `compute_information_gain()` 函数，以计算：

$$\text{Information Gain} = H(p_1^\text{node})- (w^{\text{left}}H(p_1^\text{left}) + w^{\text{right}}H(p_1^\text{right}))$$

其中：
- $H(p_1^\text{node})$ 是节点处的熵
- $H(p_1^\text{left})$ 和 $H(p_1^\text{right})$ 是划分后左分支和右分支处的熵
- $w^{\text{left}}$ 和 $w^{\text{right}}$ 分别是左分支和右分支处的样本比例

注意：
- 可以使用你在上面实现的 `compute_entropy()` 函数计算熵
- 我们提供了一些起始代码，它使用你在上面实现的 `split_dataset()` 函数拆分数据集

如果遇到困难，可以查看下方单元格之后给出的提示，以帮助你完成实现。

In [ ]:
# UNQ_C3
# GRADED FUNCTION: compute_information_gain

def compute_information_gain(X, y, node_indices, feature):
    
    """
    Compute the information of splitting the node on a given feature
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
   
    Returns:
        cost (float):        Cost computed
    
    """    
    # Split dataset
    left_indices, right_indices = split_dataset(X, node_indices, feature)
    
    # Some useful variables
    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]
    
    # You need to return the following variables correctly
    information_gain = 0
    
    ### START CODE HERE ###
    
    # Weights 
    
    #Weighted entropy
     
    #Information gain                                                   
    
    ### END CODE HERE ###  
    
    return information_gain

<details>
  <summary><font size="3" color="darkgreen"><b>单击查看提示</b></font></summary>
    
    
   * 以下是该函数整体实现的组织方式
    ```python 
    def compute_information_gain(X, y, node_indices, feature):
        # Split dataset
        left_indices, right_indices = split_dataset(X, node_indices, feature)

        # Some useful variables
        X_node, y_node = X[node_indices], y[node_indices]
        X_left, y_left = X[left_indices], y[left_indices]
        X_right, y_right = X[right_indices], y[right_indices]

        # You need to return the following variables correctly
        information_gain = 0

        ### START CODE HERE ###
        # Your code here to compute the entropy at the node using compute_entropy()
        node_entropy = 
        # Your code here to compute the entropy at the left branch
        left_entropy = 
        # Your code here to compute the entropy at the right branch
        right_entropy = 

        # Your code here to compute the proportion of examples at the left branch
        w_left = 
        
        # Your code here to compute the proportion of examples at the right branch
        w_right = 

        # Your code here to compute weighted entropy from the split using 
        # w_left, w_right, left_entropy and right_entropy
        weighted_entropy = 

        # Your code here to compute the information gain as the entropy at the node
        # minus the weighted entropy
        information_gain = 
        ### END CODE HERE ###  

        return information_gain
    ```
    如果仍然遇到困难，请查看下方提示。
    
    <details>
          <summary><font size="2" color="darkblue"><b> 计算熵的提示</b></font></summary>
        
    <code>node_entropy = compute_entropy(y_node)</code><br>
    <code>left_entropy = compute_entropy(y_left)</code><br>
    <code>right_entropy = compute_entropy(y_right)</code>
        
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 w_left 和 w_right 的提示</b></font></summary>
           <code>w_left = len(X_left) / len(X_node)</code><br>
           <code>w_right = len(X_right) / len(X_node)</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 weighted_entropy 的提示</b></font></summary>
           <code>weighted_entropy = w_left * left_entropy + w_right * right_entropy</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 information_gain 的提示</b></font></summary>
           <code> information_gain = node_entropy - weighted_entropy</code>
    </details>


</details>

现在可以使用下面的单元格检查你的实现，并计算在每个特征上进行分裂所带来的信息增益

In [ ]:
info_gain0 = compute_information_gain(X_train, y_train, root_indices, feature=0)
print("Information Gain from splitting the root on brown cap: ", info_gain0)
    
info_gain1 = compute_information_gain(X_train, y_train, root_indices, feature=1)
print("Information Gain from splitting the root on tapering stalk shape: ", info_gain1)

info_gain2 = compute_information_gain(X_train, y_train, root_indices, feature=2)
print("Information Gain from splitting the root on solitary: ", info_gain2)

# UNIT TESTS
compute_information_gain_test(compute_information_gain)

**预期输出**：
```
Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377
```

在根节点按“独居”（特征 = 2）拆分可获得最大信息增益。因此，它是根节点的最佳拆分特征。

<a name="4.4"></a>
### 4.4 获取最佳划分
现在，让我们编写一个函数，像上面那样计算每个特征的信息增益，并返回能够带来最大信息增益的特征，从而得到用于划分的最佳特征。

<a name="ex04"></a>
### 练习 4
请补全下面所示的 `get_best_split()` 函数。
- 该函数接收训练数据以及该节点上数据点的索引
- 函数输出带来最大信息增益的特征
    - 你可以使用 `compute_information_gain()` 函数遍历各个特征，并计算每个特征的信息增益
如果遇到困难，可以查看下方单元格之后给出的提示，以帮助你完成实现。

In [ ]:
# UNQ_C4
# GRADED FUNCTION: get_best_split

def get_best_split(X, y, node_indices):   
    """
    Returns the optimal feature and threshold value
    to split the node data 
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.

    Returns:
        best_feature (int):     The index of the best feature to split
    """    
    
    # Some useful variables
    num_features = X.shape[1]
    
    # You need to return the following variables correctly
    best_feature = -1
    
    ### START CODE HERE ###
       
    ### END CODE HERE ##    
   
    return best_feature

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
    
   * 下面展示了如何组织此函数的整体实现
    
    ```python 
    def get_best_split(X, y, node_indices):   

        # Some useful variables
        num_features = X.shape[1]

        # You need to return the following variables correctly
        best_feature = -1

        ### START CODE HERE ###
        max_info_gain = 0

        # Iterate through all features
        for feature in range(num_features): 
            
            # Your code here to compute the information gain from splitting on this feature
            info_gain = 
            
            # If the information gain is larger than the max seen so far
            if info_gain > max_info_gain:  
                # Your code here to set the max_info_gain and best_feature
                max_info_gain = 
                best_feature = 
        ### END CODE HERE ##    
   
    return best_feature
    ```
    如果仍然没有思路，请查看下面的提示。
    
    <details>
          <summary><font size="2" color="darkblue"><b> 计算 info_gain 的提示</b></font></summary>
        
    <code>info_gain = compute_information_gain(X, y, node_indices, feature)</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>更新 max_info_gain 和 best_feature 的提示</b></font></summary>
           <code>max_info_gain = info_gain</code><br>
           <code>best_feature = feature</code>
    </details>
</details>

现在，使用下面的单元格检查函数的实现。

In [ ]:
best_feature = get_best_split(X_train, y_train, root_indices)
print("Best feature to split on: %d" % best_feature)

# UNIT TESTS
get_best_split_test(get_best_split)

如上所示，函数返回的根节点最佳分裂特征是特征 2（“独居”）

<a name="5"></a>
## 5 - 构建树

在本节中，我们使用你在上面实现的函数生成决策树：依次选择用于划分的最佳特征，直到达到停止条件（最大深度为 2）。

这一部分无需实现任何内容。

In [ ]:
# Not graded
tree = []

def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth):
    """
    Build a tree using the recursive algorithm that split the dataset into 2 subgroups at each node.
    This function just prints the tree.
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
        branch_name (string):   Name of the branch. ['Root', 'Left', 'Right']
        max_depth (int):        Max depth of the resulting tree. 
        current_depth (int):    Current depth. Parameter used during recursive call.
   
    """ 

    # Maximum depth reached - stop splitting
    if current_depth == max_depth:
        formatting = " "*current_depth + "-"*current_depth
        print(formatting, "%s leaf node with indices" % branch_name, node_indices)
        return
   
    # Otherwise, get best split and split the data
    # Get the best feature and threshold at this node
    best_feature = get_best_split(X, y, node_indices) 
    tree.append((current_depth, branch_name, best_feature, node_indices))
    
    formatting = "-"*current_depth
    print("%s Depth %d, %s: Split on feature: %d" % (formatting, current_depth, branch_name, best_feature))
    
    # Split the dataset at the best feature
    left_indices, right_indices = split_dataset(X, node_indices, best_feature)
    
    # continue splitting the left and the right child. Increment current depth
    build_tree_recursive(X, y, left_indices, "Left", max_depth, current_depth+1)
    build_tree_recursive(X, y, right_indices, "Right", max_depth, current_depth+1)


In [ ]:
build_tree_recursive(X_train, y_train, root_indices, "Root", max_depth=2, current_depth=0)